## FASTQ-to-VCF Pipeline: Germline Variant Calling

This pipeline converts paired-end FASTQ files to a high-quality VCF using GATK Best Practices for germline short variant discovery.

### Pipeline Steps

| Step | Tool | Purpose |
|------|------|---------|
| 1. QC & Trim | `fastp` | Adapter removal, quality filtering, deduplication |
| 2. Align | `BWA-MEM2` | Map reads to reference genome |
| 3. Sort & Index | `samtools` | Coordinate-sort BAM, create index |
| 4. Mark Duplicates | `Picard` / `samtools markdup` | Flag PCR/optical duplicates |
| 5. BQSR | `GATK BaseRecalibrator + ApplyBQSR` | Correct systematic base quality errors |
| 6. Call Variants | `GATK HaplotypeCaller` | Joint-ready gVCF per-sample or direct VCF |
| 7. Filter Variants | `GATK VariantFiltration` | Hard filters for high-confidence calls |
| 8. Summary | `MultiQC` + `bcftools stats` | Aggregate QC metrics |

### Output Files
- `{sample}.final.vcf.gz` — filtered, high-quality VCF
- `{sample}.final.vcf.gz.tbi` — tabix index
- `multiqc_report.html` — aggregated QC report
- `pipeline.log` — full execution log


## Installation Requirements

### System Requirements
- **OS**: Linux (Ubuntu 20.04+ / CentOS 7+ / Debian 10+)
- **CPU**: 8+ cores recommended (16+ for WGS)
- **RAM**: 16 GB minimum, 32 GB recommended (64 GB+ for WGS)
- **Disk**: ~100 GB per WGS sample (intermediate files); SSD strongly recommended for temp files
- **Java**: JDK 17+ (required by GATK 4.x)

### Software Dependencies (install via Conda — recommended)

```bash
# 1. Create conda environment
conda create -n fq2vcf -c bioconda -c conda-forge python=3.10

# 2. Activate
conda activate fq2vcf

# 3. Install core tools
conda install -c bioconda -c conda-forge \
    bwa-mem2=2.2.1 \
    samtools=1.19 \
    bcftools=1.19 \
    fastp=0.23.4 \
    gatk4=4.5.0.0 \
    picard=3.1.1 \
    multiqc=1.21 \
    snpeff=5.1 \
    trimmomatic=0.39
```

### Alternative: Individual Tool Installation

| Tool | Version | Install Command | Notes |
|------|---------|-----------------|-------|
| BWA-MEM2 | 2.2+ | `conda install bwa-mem2` or [GitHub](https://github.com/bwa-mem2/bwa-mem2) | ~3 GB RAM for hg38 index |
| samtools | 1.17+ | `conda install samtools` | BAM processing |
| bcftools | 1.17+ | `conda install bcftools` | VCF stats/manipulation |
| fastp | 0.23+ | `conda install fastp` | QC + trimming |
| GATK | 4.5+ | `conda install gatk4` or [Broad](https://github.com/broadinstitute/gatk) | Java 17 required |
| Picard | 3.0+ | `conda install picard` | Mark duplicates (or use samtools markdup) |
| MultiQC | 1.14+ | `pip install multiqc` | QC aggregation |

### Reference Genome Files Required

```bash
# Download GRCh38 (hg38) — requires ~3 GB disk
REF_DIR="/path/to/references"
mkdir -p $REF_DIR && cd $REF_DIR

# Option A: GATK bundle (recommended)
wget -c https://storage.googleapis.com/genomics-public-data/resources/broad/hg38/v0/Homo_sapiens_assembly38.fasta
wget -c https://storage.googleapis.com/genomics-public-data/resources/broad/hg38/v0/Homo_sapiens_assembly38.fasta.fai
wget -c https://storage.googleapis.com/genomics-public-data/resources/broad/hg38/v0/Homo_sapiens_assembly38.dict

# Option B: Ensembl (GRCh38)
# https://www.ensembl.org/Homo_sapiens/Info/Index

# Build BWA-MEM2 index
bwa-mem2 index Homo_sapiens_assembly38.fasta

# Download known sites for BQSR (critical for quality)
wget -c https://storage.googleapis.com/genomics-public-data/resources/broad/hg38/v0/Homo_sapiens_assembly38.dbsnp138.vcf
wget -c https://storage.googleapis.com/genomics-public-data/resources/broad/hg38/v0/Mills_and_1000G_gold_standard.indels.hg38.vcf.gz
```

### Known Sites Files (BQSR)
| File | Source | Purpose |
|------|--------|---------|
| `dbsnp.vcf` | dbSNP (b38) | Known polymorphic sites |
| `Mills_and_1000G.indels.vcf` | 1000 Genomes | Known indels |
| `HapMap.vcf` | HapMap 3.3 | High-confidence SNP sites (for VQSR) |
